In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(dpi=100, facecolor='white')

# PreProcessing / Annotation

In [ ]:
# Direct 'raw' output from RSEM and metadata
aco_merged = ad.AnnData(pd.read_csv("Raw_skin_fracture_TPM.csv", index_col=0).T)
aco_merged.obs = pd.read_csv("Raw_skin_fracture_meta.csv", index_col=0)

In [ ]:
sc.pp.filter_cells(aco_merged, min_genes=500)
sc.pp.filter_genes(aco_merged, min_cells=1)
aco_merged.layers['TPM'] = aco_merged.X.copy()
sc.pp.log1p(aco_merged)
sc.pp.highly_variable_genes(aco_merged)
sc.tl.pca(aco_merged)

In [ ]:
sc.pl.pca_variance_ratio(aco_merged, log=True)

In [ ]:
sc.pp.neighbors(aco_merged, n_pcs=12)
sc.tl.umap(aco_merged)
sc.tl.leiden(aco_merged, resolution=0.25)

In [ ]:
sc.pl.umap(aco_merged, color=['Krt14','Pdgfra','Myh11','Ptprc','Pecam1','Lyve1','Cd68','Cd3d'
                             ,'Sox10','Mrgprb1','Il1b', 'Myh11',
                             'Cdh1','Mki67','Kit','Mitf','Tyr','Dct','Tubb3','Gap43','Mbp','Itga6','Itgb1','Itgb4','Pdgfrb','Vim',
                             ], size=15, ncols=4)

In [ ]:
sc.pl.umap(aco_merged, color=['Ptprc',
                              'Cd68','Cd86','Lyz1',
                              'S100a9','Slpi','Vegfa'
                             ], size=15, ncols=4,)

In [ ]:
new_cluster_names = "Early Macrophage(M2)/Macrophage(M2)/Epithelial/Fibroblast/Macrophage(M1)/Endothelial/T lymphocyte/Mast cell".split('/')
aco_merged.rename_categories('cell_type', new_cluster_names)

In [ ]:
# Regard M2 = Anti-inflammatory, M1= Pro-inflammatory to meet naming convention
aco_merged.obs['plot_type'] = ['Anti-inflammatory macrophages' if x in ['Early Macrophage(M2)','Macrophage(M2)'] 
                               else 'Pro-inflammatory macrophages' if x == 'Macrophage(M1)' 
                               else 'Epithelial cells' if x == 'Epithelial'
                               else 'Fibroblasts' if x == 'Fibroblast'
                               else 'Endothelial cells' if x == 'Endothelial'
                               else 'T cells' if x == 'T lymphocyte'
                               else 'Mast cells' if x == 'Mast cell'
                               else x for x in aco_merged.obs.cell_type]
plot_order = ['Pro-inflammatory macrophages', 'Anti-inflammatory macrophages', 'Epithelial cells', 'Fibroblasts',
 'Endothelial cells','T cells', 'Mast cells']
aco_merged.obs["plot_type"] = pd.Categorical(values=aco_merged.obs['plot_type'], categories=plot_order, ordered=True)

In [ ]:
aco_merged.write("skin_fracture_annotated_aco.h5ad")

In [ ]:
#Processed Files uploaded in GEO
aco_tpm = ad.AnnData(pd.read_csv("Raw_skin_fracture_TPM.csv", index_col=0).T)
aco_cnt = ad.AnnData(pd.read_csv("Raw_skin_fracture_count.csv", index_col=0).T)

aco_tpm = aco_tpm[aco_merged.obs_names]
aco_cnt = aco_cnt[aco_merged.obs_names]

aco_merged.obs.to_csv("GSE293818_skin_fracture_meta.csv")
pd.DataFrame(aco_tpm.X.T, index=aco_tpm.var.index, columns=aco_tpm.obs.index).to_csv("GSE293818_skin_fracture_TPM.csv")
pd.DataFrame(aco_cnt.X.T, index=aco_cnt.var.index, columns=aco_cnt.obs.index).to_csv("GSE293818_skin_fracture_count.csv")

## PreProcessing / Annotation for Mus

In [3]:
# Direct 'raw' output from RSEM and metadata
mus_merged = ad.AnnData(pd.read_csv("Raw_skin_mus_TPM.csv", index_col=0).T)
mus_merged.obs = pd.read_csv("Raw_skin_mus_meta.csv", index_col=0)

In [ ]:
sc.pp.filter_cells(mus_merged, min_genes=300)
sc.pp.filter_genes(mus_merged, min_cells=1)
mus_merged.layers['TPM'] = mus_merged.X.copy()
sc.pp.log1p(mus_merged)
sc.pp.highly_variable_genes(mus_merged)
sc.tl.pca(mus_merged)

In [ ]:
sc.pl.pca_variance_ratio(mus_merged, log=True)

In [ ]:
sc.pp.neighbors(mus_merged, n_pcs=15) 
sc.tl.umap(mus_merged)
sc.tl.leiden(mus_merged, resolution=0.32)

In [ ]:
sc.pl.umap(mus_merged, color=['Krt14','Pdgfra','Myh11','Ptprc','Pecam1','Lyve1','Cd68','Cd3d'
                             ,'Sox10','Mrgprb1','Il1b', 'Myh11',
                             'Cdh1','Mki67','Kit','Mitf','Tyr','Dct','Tubb3','Gap43','Mbp','Itga6','Itgb1','Itgb4','Pdgfrb','Vim',
                              'Myh8','Pdk4','Myog'
                             ], size=20, ncols=4, use_raw=False)

In [ ]:
sc.pl.umap(mus_merged, color=['Ptprc',
                              'Cd68','Cd86','Lyz1',
                              'S100a9','Slpi','Vegfa'
                             ], size=15, ncols=4,use_raw=False)

In [ ]:
mus_merged.obs['cell_type'] = mus_merged.obs['leiden']
new_cluster_names = "Pro-inflammatory Macrophages(M1)/Anti-inflammatory Macrophages(M2)/Fibroblasts/Epithelial cells/Endothelial cells/Myofibroblasts".split('/')
mus_merged.rename_categories('cell_type', new_cluster_names)

mus_merged.obs['base_type'] = mus_new.obs['cell_type']

In [ ]:
mus_merged.write("skin_fracture_annotated_mus.h5ad")

In [ ]:
#Processed Files uploaded in GEO
mus_tpm = ad.AnnData(pd.read_csv("Raw_skin_mus_TPM.csv", index_col=0).T)
mus_cnt = ad.AnnData(pd.read_csv("Raw_skin_mus_count.csv", index_col=0).T)

mus_tpm = mus_tpm[mus_merged.obs_names]
mus_cnt = mus_cnt[mus_merged.obs_names]

mus_merged.obs.to_csv("GSE293818_skin_mus_meta.csv")
pd.DataFrame(mus_tpm.X.T, index=mus_tpm.var.index, columns=mus_tpm.obs.index).to_csv("GSE293818_skin_mus_TPM.csv")
pd.DataFrame(mus_cnt.X.T, index=mus_cnt.var.index, columns=mus_cnt.obs.index).to_csv("GSE293818_skin_mus_count.csv")

# Analysis

## QC metrics

In [ ]:
aco_merged = sc.read("skin_fracture_annotated_aco.h5ad")

In [ ]:
raw_counts = ad.AnnData(pd.read_csv("Raw_skin_fracture_count.csv", index_col=0).T)
raw_counts.obs = pd.read_csv("Raw_skin_fracture_meta.csv", index_col=0)

In [ ]:
sc.pp.calculate_qc_metrics(raw_counts, percent_top=None, inplace=True)

In [ ]:
# Unfiltered count
plt.rcParams["figure.figsize"] = (10,10)
plt.rcParams['figure.dpi'] = 1000
sc.pl.violin(raw_counts, ['total_counts'], groupby='wound_fracture', log=True, save='Unfiltered_raw_counts.png')
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]
plt.rcParamsDefault['figure.dpi'] = plt.rcParamsDefault['figure.dpi']

In [ ]:
# Unfiltered count
plt.rcParams["figure.figsize"] = (10,10)
plt.rcParams['figure.dpi'] = 1000
sc.pl.violin(raw_counts, ['n_genes_by_counts'], groupby='wound_fracture', log=False, save='Unfiltered_raw_gene_counts.png')
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]
plt.rcParamsDefault['figure.dpi'] = plt.rcParamsDefault['figure.dpi']

In [ ]:
#filtered
raw_filtered = raw_counts[aco_merged.obs_names, :].copy()

In [ ]:
# filtered count
plt.rcParams["figure.figsize"] = (10,10)
plt.rcParams['figure.dpi'] = 1000
sc.pl.violin(raw_filtered, ['total_counts'], groupby='wound_fracture', log=True, save='Filtered_raw_counts.png')
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]
plt.rcParamsDefault['figure.dpi'] = plt.rcParamsDefault['figure.dpi']

In [ ]:
# filtered count
plt.rcParams["figure.figsize"] = (10,10)
plt.rcParams['figure.dpi'] = 1000
sc.pl.violin(raw_filtered, ['n_genes_by_counts'], groupby='wound_fracture', log=False, save='Filtered_raw_gene_counts.png')
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]
plt.rcParamsDefault['figure.dpi'] = plt.rcParamsDefault['figure.dpi']

### Mus

In [ ]:
mus_merged = sc.read("../adata/skin_fracture_annotated_mus.h5ad")

In [ ]:
raw_counts = ad.AnnData(pd.read_csv("Raw_skin_mus_count.csv", index_col=0).T)
raw_counts.obs = pd.read_csv("Raw_skin_mus_meta.csv", index_col=0)

In [ ]:
sc.pp.calculate_qc_metrics(raw_counts, percent_top=None, inplace=True)

In [ ]:
# Unfiltered count
plt.rcParams["figure.figsize"] = (10,10)
plt.rcParams['figure.dpi'] = 1000
sc.pl.violin(raw_counts, ['total_counts'], groupby='wound', log=True, save='Mus_Unfiltered_raw_counts.png')
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]
plt.rcParamsDefault['figure.dpi'] = plt.rcParamsDefault['figure.dpi']

In [ ]:
# Unfiltered count
plt.rcParams["figure.figsize"] = (10,10)
plt.rcParams['figure.dpi'] = 1000
sc.pl.violin(raw_counts, ['n_genes_by_counts'], groupby='wound', log=False, save='Mus_Unfiltered_raw_gene_counts.png')
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]
plt.rcParamsDefault['figure.dpi'] = plt.rcParamsDefault['figure.dpi']

In [ ]:
#filtered
raw_filtered = raw_counts[mus_merged.obs_names, :].copy()

In [ ]:
# filtered count
plt.rcParams["figure.figsize"] = (10,10)
plt.rcParams['figure.dpi'] = 1000
sc.pl.violin(raw_filtered, ['total_counts'], groupby='wound', log=True, save='Mus_Filtered_raw_counts.png')
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]
plt.rcParamsDefault['figure.dpi'] = plt.rcParamsDefault['figure.dpi']

In [ ]:
# filtered count
plt.rcParams["figure.figsize"] = (10,10)
plt.rcParams['figure.dpi'] = 1000
sc.pl.violin(raw_filtered, ['n_genes_by_counts'], groupby='wound', log=False, save='Mus_Filtered_raw_gene_counts.png')
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]
plt.rcParamsDefault['figure.dpi'] = plt.rcParamsDefault['figure.dpi']

## Plotting

### Fig 5F : UMAP plot

In [ ]:
plt.rcParams["figure.figsize"] = (10,10)
plt.rcParams['figure.dpi'] = 1000
for f in ['plot_type', 'wound_fracture','wound','fracture']:
    sc.pl.umap(aco_merged, color=[f], size=30, use_raw=False, save='250618_Fig'+f+'.png',show=False)
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]
plt.rcParamsDefault['figure.dpi'] = plt.rcParamsDefault['figure.dpi']

In [ ]:
plt.rcParams["figure.figsize"] = (10,10)
plt.rcParams['figure.dpi'] = 1000
for f in ["PW6h", "PWD2", "PWD6"]:
    aco_merged.obs['temp'] = [aco_merged.obs.fracture[i] if aco_merged.obs.wound[i] == f else None for i in range(len(aco_merged.obs))]
    sc.pl.umap(aco_merged, color=['temp'], size=60, title=f, save='WithBackground_Fig'+f+'.png', show=False, na_color='gainsboro')
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]
plt.rcParamsDefault['figure.dpi'] = plt.rcParamsDefault['figure.dpi']

### Fig 5G : DEG dot plot

In [ ]:
fib = aco_merged[(aco_merged.obs.wound=='PWD2')&(aco_merged.obs.cell_type.str.contains('ibroblast'))].copy()
epi = aco_merged[(aco_merged.obs.wound=='PWD2')&(aco_merged.obs.cell_type.str.contains('pithelial'))].copy()
mac = aco_merged[(aco_merged.obs.wound=='PWD2')&(aco_merged.obs.cell_type.str.contains('acrophage'))].copy()

In [ ]:
fib.obs['fracture_name'] = ["Fibroblast_"+x for x in fib.obs['fracture']]
dp = sc.pl.dotplot(fib, ['Pdgfra',
                   'Eif3i','Eif4a1','Aimp2','Pgam1','Tpi1','Pkm','Itgav','Actg1','Cav1','Cd151',
                   'Col1a1','Col1a2','Col3a1','Col15a1','Fbln1','Eln','Vit','Mmp2','Mmp23','Vegfc','Igf1'], 'fracture', show=False)
ax = dp["mainplot_ax"]
for l in ax.get_xticklabels():
    l.set_style("italic")
    l.set_fontsize(l.get_fontsize()*1.3)
    
plt.savefig('Italic_Fib_PWD2.png', dpi=1000, bbox_inches='tight')

In [ ]:
fib.obs['fracture_name'] = ["Fibroblast_"+x for x in fib.obs['fracture']]
dp = sc.pl.dotplot(epi, ['Cdh1','Epcam',
                   'Dlx1','Cux1','Cebpb','Lbh','Lef1','Msx2','Lhx2','Kdm5a','Slc39a6','Slc39a10','Slc7a8','Egfr',
                   'Ccl8','Ccl27a','Il7','Il18','Il33','Cxcl17','Il1r2','Il20rb'], 'fracture', show=False)
ax = dp["mainplot_ax"]
for l in ax.get_xticklabels():
    l.set_style("italic")
    l.set_fontsize(l.get_fontsize()*1.3)
    
plt.savefig('Italic_Epi_PWD2.png', dpi=1000, bbox_inches='tight')

In [ ]:
mac.obs['fracture_name'] = ["Macrophage_"+x for x in mac.obs['fracture']]
dp = sc.pl.dotplot(mac, ['Ptprc','Itgam','Mertk',
                   'S100a9','Il1r2','Ccl3','Cxcl2','Slpi','Ccrl2','Vegfa','Ltf','Adam8',
                   'Cd68','Cd36','Ccl8','Lyz1','Rpl7','Rps8','Rpl36a','Ctss','Cd74','Ifi30',], 'fracture', show=False)
ax = dp["mainplot_ax"]
for l in ax.get_xticklabels():
    l.set_style("italic")
    l.set_fontsize(l.get_fontsize()*1.3)
    
plt.savefig('Italic_Mac_PWD2.png', dpi=1000, bbox_inches='tight')

### Fig S8A : Marker Genes UMAP plot

In [ ]:
plt.rcParams["figure.figsize"] = (10,10)
ax = sc.pl.umap(aco_merged, color=['S100a9','Slpi',
                              'Vegfa','Cd68',
                              'Lyz1','Cd86',
                             'Cd4','Cd3d',
                             'Kit','Mrgprb1'], size=30, ncols=2, show=False)
plt.rcParamsDefault['figure.dpi'] = plt.rcParamsDefault['figure.dpi']

In [ ]:
for i in range(10):
    l = ax.axes[i*2]
    l.set_title(l.get_title(), 
                    {'fontsize':'x-large',
                    'fontstyle':'italic'})
ax.savefig('Marker_genes_immune.png', dpi=1000, bbox_inches='tight')

In [ ]:
plt.rcParams["figure.figsize"] = (10,10)
ax = sc.pl.umap(aco_merged, color=['Epcam','Cdh1',
                             'Pdgfra','Fap',
                             'Pecam1','Cdh5',], size=30, ncols=2, show=False, return_fig=True)
plt.rcParamsDefault['figure.dpi'] = plt.rcParamsDefault['figure.dpi']

In [ ]:
for i in range(6):
    l = ax.axes[i*2]
    l.set_title(l.get_title(), 
                    {'fontsize':'x-large',
                    'fontstyle':'italic'})
ax.savefig('Marker_genes_others.png', dpi=1000, bbox_inches='tight')

### Fig S8B : DEG dot plot

In [ ]:
mac_all = aco_merged[(aco_merged.obs.plot_type.str.contains('macrophage'))].copy()
mac_all.obs['mac'] = mac_all.obs.plot_type
sc.tl.rank_genes_groups(mac_all, 'plot_type')
df_all = pd.DataFrame(mac_all.uns['rank_genes_groups']['names'])

In [ ]:
dp = sc.pl.dotplot(mac_all, 
    dict(pd.concat([df_all.iloc[[0,1,2,3,4,5,7,8,9,10,11,12,13,14,15,16,17,19],0].reset_index(),
          df_all.iloc[[0,1,2,3,4,5,6,7,9,10,11,12,13,14,15,16,17,18],1].reset_index()],axis=1).loc[:,['Pro-inflammatory macrophages','Anti-inflammatory macrophages']]), 
                   'plot_type', show=False,
                  categories_order=['Pro-inflammatory macrophages','Anti-inflammatory macrophages'],
                  var_group_rotation=0)
ax = dp["mainplot_ax"]
for l in ax.get_xticklabels():
    l.set_style("italic")
    l.set_fontsize(l.get_fontsize()*1.3)

plt.savefig('250618_Mac_DEG_dot_plot.png', dpi=1000, bbox_inches='tight')

### Fig S8C : Cell Composion Bar plot

In [ ]:
plt.cla()
com_plot = (aco_merged.obs[['wound_fracture','plot_type']].groupby('wound_fracture').value_counts(normalize=True)*100).unstack().plot(kind='bar',stacked=True)
com_plot.legend(loc='center left', bbox_to_anchor=(1,0.5))
plt.savefig('250618_Cell_by_woundxfracture.png', dpi=1000, bbox_inches='tight')

### Fig S@ : Mus Plots

In [ ]:
plt.rcParams["figure.figsize"] = (10,10)
plt.rcParams['figure.dpi'] = 1000
for f in ['cell_type',  'wound']:
    sc.pl.umap(mus_merged, color=[f], size=60, use_raw=False, save='Mus_260223_Fig'+f+'_bigdot.png',show=False)
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]
plt.rcParamsDefault['figure.dpi'] = plt.rcParamsDefault['figure.dpi']

In [ ]:
plt.rcParams["figure.figsize"] = (10,10)
plt.rcParams['figure.dpi'] = 1000
for f in ["PW6h", "PWD2", "PWD6"]:
    mus_merged.obs['temp'] = [f if i == f else None for i in mus_merged.obs.wound]
    sc.pl.umap(mus_merged, color=['temp'], size=60, title=f, save='Mus_WithBackground_Fig'+f+'.png', show=False, na_color='gainsboro')
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]
plt.rcParamsDefault['figure.dpi'] = plt.rcParamsDefault['figure.dpi']

In [ ]:
dp = sc.pl.dotplot(mus_merged[mus_merged.obs.cell_type=='Fibroblasts'], ['Pdgfra',
                   'Eif3i','Eif4a1','Aimp2','Pgam1','Tpi1','Pkm','Itgav','Actg1','Cav1','Cd151',
                   'Col1a1','Col1a2','Col3a1','Col15a1','Fbln1','Eln','Vit','Mmp2','Mmp23','Vegfc','Igf1'], 'wound', show=False,
                   dot_max=1, dot_min=0)
ax = dp["mainplot_ax"]
for l in ax.get_xticklabels():
    l.set_style("italic")
    l.set_fontsize(l.get_fontsize()*1.3)
    
plt.savefig('Mus_Italic_Fib.png', dpi=1000, bbox_inches='tight')

In [ ]:
dp = sc.pl.dotplot(mus_merged[mus_merged.obs.cell_type=='Epithelial cells'], ['Cdh1','Epcam',
                   'Dlx1','Cux1','Cebpb','Lbh','Lef1','Msx2','Lhx2','Kdm5a','Slc39a6','Slc39a10','Slc7a8','Egfr',
                   'Ccl8','Ccl27a','Il7','Il18','Il33','Cxcl17','Il1r2','Il20rb'], 'wound', show=False,
                   dot_max=1, dot_min=0)
ax = dp["mainplot_ax"]
for l in ax.get_xticklabels():
    l.set_style("italic")
    l.set_fontsize(l.get_fontsize()*1.3)
    
plt.savefig('Mus_Italic_Epi.png', dpi=1000, bbox_inches='tight')

In [ ]:
dp = sc.pl.dotplot(mus_merged[mus_merged.obs.cell_type.str.contains('Macrophage')], ['Ptprc','Itgam','Mertk',
                   'S100a9','Il1r2','Ccl3','Cxcl2','Slpi','Ccrl2','Vegfa','Ltf','Adam8',
                   'Cd68','Cd36','Ccl8','Lyz1','Rpl7','Rps8','Rpl36a','Ctss','Cd74','Ifi30',], 'wound', show=False,
                   dot_max=1, dot_min=0)
ax = dp["mainplot_ax"]
for l in ax.get_xticklabels():
    l.set_style("italic")
    l.set_fontsize(l.get_fontsize()*1.3)
    
plt.savefig('Mus_Italic_Mac.png', dpi=1000, bbox_inches='tight')

In [ ]:
plt.rcParams["figure.figsize"] = (10,10)
ax = sc.pl.umap(mus_merged, color=['S100a9','Slpi',
                              'Vegfa','Cd68',
                              'Lyz1','Cd86',
                             'Cd4','Cd3d',
                             'Kit','Mrgprb1'], size=60, ncols=2, use_raw=False, show=False, return_fig=True)
plt.rcParamsDefault['figure.dpi'] = plt.rcParamsDefault['figure.dpi']

In [ ]:
for i in range(10):
    l = ax.axes[i*2]
    l.set_title(l.get_title(), 
                    {'fontsize':'x-large',
                    'fontstyle':'italic'})
ax.savefig('Mus_Marker_genes_immune.png', dpi=1000, bbox_inches='tight')

In [ ]:
plt.rcParams["figure.figsize"] = (10,10)
ax = sc.pl.umap(mus_merged, color=['Epcam','Cdh1',
                             'Pdgfra','Fap',
                             'Pecam1','Cdh5',], size=60, ncols=2, use_raw=False, show=False, return_fig=True)
plt.rcParamsDefault['figure.dpi'] = plt.rcParamsDefault['figure.dpi']

In [ ]:
for i in range(6):
    l = ax.axes[i*2]
    l.set_title(l.get_title(), 
                    {'fontsize':'x-large',
                    'fontstyle':'italic'})
ax.savefig('Mus_Marker_genes_others.png', dpi=1000, bbox_inches='tight')

In [ ]:
plt.cla()
com_plot = (mus_merged.obs[['wound','cell_type']].groupby('wound').value_counts(normalize=True)*100).unstack().plot(kind='bar',stacked=True)
com_plot.legend(loc='center left', bbox_to_anchor=(1,0.5))
plt.savefig('Mus_260223_Cell_by_wound.png', dpi=1000, bbox_inches='tight')

### Mus-Aco comparison

#### Variable-normalize (gene-normalize)

In [ ]:
dp = sc.pl.dotplot(mus_merged[mus_merged.obs.cell_type=='Fibroblasts'], ['Pdgfra',
                   'Eif3i','Eif4a1','Aimp2','Pgam1','Tpi1','Pkm','Itgav','Actg1','Cav1','Cd151',
                   'Col1a1','Col1a2','Col3a1','Col15a1','Fbln1','Eln','Vit','Mmp2','Mmp23','Vegfc','Igf1'], 'wound', show=False,
                   dot_max=1, dot_min=0, standard_scale='var')
ax = dp["mainplot_ax"]
for l in ax.get_xticklabels():
    l.set_style("italic")
    l.set_fontsize(l.get_fontsize()*1.3)
    
plt.savefig('Mus_Italic_Fib_var_normalized.png', dpi=1000, bbox_inches='tight')

In [ ]:
dp = sc.pl.dotplot(mus_merged[mus_merged.obs.cell_type=='Epithelial cells'], ['Cdh1','Epcam',
                   'Dlx1','Cux1','Cebpb','Lbh','Lef1','Msx2','Lhx2','Kdm5a','Slc39a6','Slc39a10','Slc7a8','Egfr',
                   'Ccl8','Ccl27a','Il7','Il18','Il33','Cxcl17','Il1r2','Il20rb'], 'wound', show=False,
                   dot_max=1, dot_min=0, standard_scale='var')
ax = dp["mainplot_ax"]
for l in ax.get_xticklabels():
    l.set_style("italic")
    l.set_fontsize(l.get_fontsize()*1.3)
    
plt.savefig('Mus_Italic_Epi_var_normalized.png', dpi=1000, bbox_inches='tight')

In [ ]:
dp = sc.pl.dotplot(mus_merged[mus_merged.obs.cell_type.str.contains('Macrophage')], ['Ptprc','Itgam','Mertk',
                   'S100a9','Il1r2','Ccl3','Cxcl2','Slpi','Ccrl2','Vegfa','Ltf','Adam8',
                   'Cd68','Cd36','Ccl8','Lyz1','Rpl7','Rps8','Rpl36a','Ctss','Cd74','Ifi30',], 'wound', show=False,
                   dot_max=1, dot_min=0, standard_scale='var')
ax = dp["mainplot_ax"]
for l in ax.get_xticklabels():
    l.set_style("italic")
    l.set_fontsize(l.get_fontsize()*1.3)
    
plt.savefig('Mus_Italic_Mac_var_normalized.png', dpi=1000, bbox_inches='tight')

In [ ]:
dp = sc.pl.dotplot(aco_merged[(aco_merged.obs.wound=='PWD2')&(aco_merged.obs.base_type=='Fibroblasts')], ['Pdgfra',
                   'Eif3i','Eif4a1','Aimp2','Pgam1','Tpi1','Pkm','Itgav','Actg1','Cav1','Cd151',
                   'Col1a1','Col1a2','Col3a1','Col15a1','Fbln1','Eln','Vit','Mmp2','Mmp23','Vegfc','Igf1'], 'fracture', show=False,
                   dot_max=1, dot_min=0, standard_scale='var')
ax = dp["mainplot_ax"]
for l in ax.get_xticklabels():
    l.set_style("italic")
    l.set_fontsize(l.get_fontsize()*1.3)
    
plt.savefig('260223_Aco_Italic_Fib_var_normalized.png', dpi=1000, bbox_inches='tight')

In [ ]:
dp = sc.pl.dotplot(aco_merged[(aco_merged.obs.wound=='PWD2')&(aco_merged.obs.base_type=='Epithelial cells')], ['Cdh1','Epcam',
                   'Dlx1','Cux1','Cebpb','Lbh','Lef1','Msx2','Lhx2','Kdm5a','Slc39a6','Slc39a10','Slc7a8','Egfr',
                   'Ccl8','Ccl27a','Il7','Il18','Il33','Cxcl17','Il1r2','Il20rb'], 'fracture', show=False,
                   dot_max=1, dot_min=0, standard_scale='var')
ax = dp["mainplot_ax"]
for l in ax.get_xticklabels():
    l.set_style("italic")
    l.set_fontsize(l.get_fontsize()*1.3)
    
plt.savefig('260223_Aco_Italic_Epi_var_normalized.png', dpi=1000, bbox_inches='tight')

In [ ]:
dp = sc.pl.dotplot(aco_merged[(aco_merged.obs.wound=='PWD2')&(aco_merged.obs.base_type.str.contains('Macrophage'))], ['Ptprc','Itgam','Mertk',
                   'S100a9','Il1r2','Ccl3','Cxcl2','Slpi','Ccrl2','Vegfa','Ltf','Adam8',
                   'Cd68','Cd36','Ccl8','Lyz1','Rpl7','Rps8','Rpl36a','Ctss','Cd74','Ifi30',], 'fracture', show=False,
                   dot_max=1, dot_min=0, standard_scale='var')
ax = dp["mainplot_ax"]
for l in ax.get_xticklabels():
    l.set_style("italic")
    l.set_fontsize(l.get_fontsize()*1.3)
    
plt.savefig('260223_Aco_Italic_Mac_var_normalized.png', dpi=1000, bbox_inches='tight')

#### Aco genes as score

In [ ]:
score_list ={
   'fib_rip' : ['Eif3i','Eif4a1','Aimp2','Pgam1','Tpi1','Pkm','Itgav','Actg1','Cav1','Cd151'],
    'fib_cut' : ['Col1a1','Col1a2','Col3a1','Col15a1','Fbln1','Eln','Vit','Mmp2','Mmp23','Vegfc','Igf1'],
    'epi_rip' : ['Dlx1','Cux1','Cebpb','Lbh','Lef1','Msx2','Lhx2','Kdm5a','Slc39a6','Slc39a10','Slc7a8','Egfr'],
    'epi_cut' : ['Ccl8','Ccl27a','Il7','Il18','Il33','Cxcl17','Il1r2','Il20rb'],
    'mac_rip' : ['S100a9','Il1r2','Ccl3','Cxcl2','Slpi','Ccrl2','Vegfa','Ltf','Adam8'],
    'mac_cut' : ['Cd68','Cd36','Ccl8','Lyz1','Rpl7','Rps8','Rpl36a','Ctss','Cd74','Ifi30'] 
}


In [ ]:
for k, v in score_list.items():
    sc.tl.score_genes(aco_merged, v, score_name=k)
    sc.tl.score_genes(mus_merged, v, score_name=k)

In [ ]:
df = pd.concat([aco_merged.obs[['base_type','wound', 'fracture']+list(score_list.keys())].copy(), 
                mus_merged.obs[['base_type','wound', 'fracture']+list(score_list.keys())].copy()], 
               ignore_index=True)

time_map = {
    'PW6h': 1.0,
    'PWD2': 2.0,
    'PWD6': 3.0
}
df['wound_num'] = df['wound'].map(time_map).astype(float)

In [ ]:
for sco in ['mac_rip','mac_cut']:
    plt.cla()
    g = sns.lineplot(
        data=df[df.base_type.str.contains('Macrophage')], 
        x='wound_num', 
        y=sco,
        hue='fracture', 
        marker='o',       
        linewidth=2.5,
        errorbar='ci'
    )
    
    inv_map = {v: k for k, v in time_map.items()}
    ticks = sorted(time_map.values())
    labels = [inv_map[t] for t in ticks]
    
    plt.xticks(ticks, labels)
    
    plt.title(f'Temporal Trend of {sco}')
    plt.xlabel('Timepoint')
    plt.ylabel('Score')
        
    plt.tight_layout()
    plt.savefig(f'260223_Gene_score_{sco}.png', dpi=1000, bbox_inches='tight')

In [ ]:
for sco in ['fib_rip','fib_cut']:
    plt.cla()
    g = sns.lineplot(
        data=df[df.base_type.str.contains('Fibroblasts')], 
        x='wound_num', 
        y=sco,
        hue='fracture', 
        marker='o',       
        linewidth=2.5,
        errorbar='ci'
    )
    
    inv_map = {v: k for k, v in time_map.items()}
    ticks = sorted(time_map.values())
    labels = [inv_map[t] for t in ticks]
    
    plt.xticks(ticks, labels)
    
    plt.title(f'Temporal Trend of {sco}')
    plt.xlabel('Timepoint')
    plt.ylabel('Score')
        
    plt.tight_layout()
    plt.savefig(f'260223_Gene_score_{sco}.png', dpi=1000, bbox_inches='tight')

In [ ]:
for sco in ['epi_rip','epi_cut']:
    plt.cla()
    g = sns.lineplot(
        data=df[df.base_type.str.contains('Fibroblasts')], 
        x='wound_num', 
        y=sco,
        hue='fracture', 
        marker='o',       
        linewidth=2.5,
        errorbar='ci'
    )
    
    inv_map = {v: k for k, v in time_map.items()}
    ticks = sorted(time_map.values())
    labels = [inv_map[t] for t in ticks]
    
    plt.xticks(ticks, labels)
    
    plt.title(f'Temporal Trend of {sco}')
    plt.xlabel('Timepoint')
    plt.ylabel('Score')
        
    plt.tight_layout()
    plt.savefig(f'260223_Gene_score_{sco}.png', dpi=1000, bbox_inches='tight')

#### Mus-Aco merging

In [ ]:
var_names = aco_merged.var_names.intersection(mus_merged.var_names)
aco_merged = aco_merged[:, var_names].copy()
mus_merged = mus_merged[:, var_names].copy()

In [ ]:
mus_merged.obs['orig_base_type'] = mus_merged.obs['base_type']
sc.tl.ingest(mus_merged, aco_merged, obs='base_type')

In [ ]:
all_merged = ad.concat([aco_merged, mus_merged], label='species', keys=['aco','mus'])

In [ ]:
# Fix ordering
all_merged.obs['fracture'] = all_merged.obs['fracture'].astype('category')
all_merged.obs['fracture'] = all_merged.obs['fracture'].cat.reorder_categories(['Hand','Scissor','Mus'], ordered=True)

In [ ]:
sc.pl.umap(all_merged, color=['cell_type','base_type','wound','fracture'],ncols=2)

In [ ]:
plt.rcParams["figure.figsize"] = (10,10)
plt.rcParams['figure.dpi'] = 1000
for f in ['cell_type', 'base_type','wound','fracture','species']:
    sc.pl.umap(all_merged, color=[f], size=30, use_raw=False, save='All_260223_Fig'+f+'.png',show=False)
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]
plt.rcParamsDefault['figure.dpi'] = plt.rcParamsDefault['figure.dpi']

In [ ]:
plt.rcParams["figure.figsize"] = (10,10)
plt.rcParams['figure.dpi'] = 1000
for f in ["PW6h", "PWD2", "PWD6"]:
    all_merged.obs['temp'] = [all_merged.obs.fracture[i] if all_merged.obs.wound[i] == f else None for i in range(len(all_merged.obs))]
    sc.pl.umap(all_merged, color=['temp'], size=60, title=f, save='All_WithBackground_Fig'+f+'.png', show=False, na_color='gainsboro')
plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]
plt.rcParamsDefault['figure.dpi'] = plt.rcParamsDefault['figure.dpi']

In [ ]:
dp = sc.pl.dotplot(all_merged[(all_merged.obs.wound=='PWD2')&(all_merged.obs.base_type=='Fibroblasts')], ['Pdgfra',
                   'Eif3i','Eif4a1','Aimp2','Pgam1','Tpi1','Pkm','Itgav','Actg1','Cav1','Cd151',
                   'Col1a1','Col1a2','Col3a1','Col15a1','Fbln1','Eln','Vit','Mmp2','Mmp23','Vegfc','Igf1'], 'fracture', show=False,
                   dot_max=1, dot_min=0)
ax = dp["mainplot_ax"]
for l in ax.get_xticklabels():
    l.set_style("italic")
    l.set_fontsize(l.get_fontsize()*1.3)
    
plt.savefig('260223_All_Italic_Fib.png', dpi=1000, bbox_inches='tight')

In [ ]:
dp = sc.pl.dotplot(all_merged[(all_merged.obs.wound=='PWD2')&(all_merged.obs.base_type=='Epithelial cells')], ['Cdh1','Epcam',
                   'Dlx1','Cux1','Cebpb','Lbh','Lef1','Msx2','Lhx2','Kdm5a','Slc39a6','Slc39a10','Slc7a8','Egfr',
                   'Ccl8','Ccl27a','Il7','Il18','Il33','Cxcl17','Il1r2','Il20rb'], 'fracture', show=False,
                   dot_max=1, dot_min=0)
ax = dp["mainplot_ax"]
for l in ax.get_xticklabels():
    l.set_style("italic")
    l.set_fontsize(l.get_fontsize()*1.3)
    
plt.savefig('260223_All_Italic_Epi.png', dpi=1000, bbox_inches='tight')

In [ ]:
dp = sc.pl.dotplot(all_merged[(all_merged.obs.wound=='PWD2')&(all_merged.obs.base_type.str.contains('Macrophage'))], ['Ptprc','Itgam','Mertk',
                   'S100a9','Il1r2','Ccl3','Cxcl2','Slpi','Ccrl2','Vegfa','Ltf','Adam8',
                   'Cd68','Cd36','Ccl8','Lyz1','Rpl7','Rps8','Rpl36a','Ctss','Cd74','Ifi30',], 'fracture', show=False,
                   dot_max=1, dot_min=0)
ax = dp["mainplot_ax"]
for l in ax.get_xticklabels():
    l.set_style("italic")
    l.set_fontsize(l.get_fontsize()*1.3)
    
plt.savefig('260223_All_Italic_Mac.png', dpi=1000, bbox_inches='tight')

In [ ]:
# Checking mus/aco specific degs
mac = all_merged[(all_merged.obs.base_type.str.contains('Macrophage')) & (all_merged.obs.wound=='PWD2')]
sc.tl.rank_genes_groups(mac, 'species')
sc.pl.dotplot(mac, dict(pd.DataFrame(
    mac.uns['rank_genes_groups']['names']).head(20)), 'species', save='Compare_mac_top20_PWD2.png')

In [ ]:
# Checking mus/aco specific degs
fib = all_merged[(all_merged.obs.base_type.str.contains('Fibroblasts')) & (all_merged.obs.wound=='PWD2')]
sc.tl.rank_genes_groups(fib, 'species')
sc.pl.dotplot(fib, dict(pd.DataFrame(
    fib.uns['rank_genes_groups']['names']).head(20)), 'species', save='Compare_fib_top20_PWD2.png')

In [ ]:
# Checking mus/aco specific degs
epi = all_merged[(all_merged.obs.base_type.str.contains('Epithelial')) & (all_merged.obs.wound=='PWD2')]
sc.tl.rank_genes_groups(epi, 'species')
sc.pl.dotplot(epi, dict(pd.DataFrame(
    epi.uns['rank_genes_groups']['names']).head(20)), 'species', save='Compare_epi_top20_PWD2.png')

In [ ]:
score_list ={
    'mac_aco' : ['Gm8909','Prdx6b','Chil1','Fkbp11','Saa1','Saa2','Upp1','Agbl5','Serpinb1b','Arl4a'],
    'mac_mus' : ['Lars2','Cd52','Wfdc17','Ifi27l2a','Ifitm2','Ccl9','Slfn2','Plac8','C1qb','Marcksl1'] 
}

In [ ]:
for k, v in score_list.items():
    print(sum(x in ['S100a9','Il1r2','Ccl3','Cxcl2','Slpi','Ccrl2','Vegfa','Ltf','Adam8',
                    'Cd68','Cd36','Ccl8','Lyz1','Rpl7','Rps8','Rpl36a','Ctss','Cd74','Ifi30'] for x in v))

In [ ]:
aco_merged = sc.read("skin_fracture_annotated_aco.h5ad")
mus_merged = sc.read("skin_fracture_annotated_mus.h5ad")

In [ ]:
for k, v in score_list.items():
    sc.tl.score_genes(aco_merged, v, score_name=k)
    sc.tl.score_genes(mus_merged, v, score_name=k)

In [ ]:
df = pd.concat([aco_merged.obs[['base_type','wound', 'fracture']+list(score_list.keys())].copy(), 
                mus_merged.obs[['base_type','wound', 'fracture']+list(score_list.keys())].copy()], 
               ignore_index=True)

time_map = {
    'PW6h': 1.0,
    'PWD2': 2.0,
    'PWD6': 3.0
}
df['wound_num'] = df['wound'].map(time_map).astype(float)

In [ ]:
plt.rcParams["figure.figsize"] = (10,10)
plt.rcParams['figure.dpi'] = 1000
for sco in ['mac_mus','mac_aco']:
    plt.cla()
    g = sns.lineplot(
        data=df[df.base_type.str.contains('Macrophage')], 
        x='wound_num', 
        y=sco,
        hue='fracture', 
        marker='o',       
        linewidth=2.5,
        errorbar='ci'
    )
    
    inv_map = {v: k for k, v in time_map.items()}
    ticks = sorted(time_map.values())
    labels = [inv_map[t] for t in ticks]
    
    plt.xticks(ticks, labels)
    
    plt.title(f'Temporal Trend of {sco}')
    plt.xlabel('Timepoint')
    plt.ylabel('Score')
        
    plt.tight_layout()
    plt.savefig(f'260223_Gene_score_{sco}.png', dpi=1000, bbox_inches='tight')